[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_07_RAG.ipynb)

# 📚 Lesson 7: RAG — Retrieval-Augmented Generation

**Your AI Learning Journey | Lesson 7 of 9**

---

## The Big Problem We're Solving

Imagine you built a customer support agent for your company. You ask it:

> *"What is our refund policy for purchases made after March 2026?"*

The LLM will either:
- **Hallucinate** a policy that sounds plausible but is wrong
- Say **"I don't know"** because this info wasn't in its training data

**Why?** LLMs are trained once, at a point in time. They have no access to:
- Your private company docs
- Your latest policies
- Anything that changed after their training cutoff

**RAG solves this.** It's the technique that makes LLMs useful on YOUR data.

---

## What You'll Build Today

A complete RAG pipeline:
1. **Embed** a knowledge base of documents into vectors
2. **Store** them in a vector database (ChromaDB)
3. **Retrieve** relevant chunks when a question comes in
4. **Generate** an answer using Claude + the retrieved context

By the end, you'll have a mini "ask questions about your docs" system.

---

## The RAG Architecture at a Glance

```
INDEXING PHASE (done once, offline)
─────────────────────────────────────────────────────────
Documents → [Chunker] → Chunks → [Embedder] → Vectors → [Vector DB]

QUERY PHASE (done at runtime, per question)
─────────────────────────────────────────────────────────
Question → [Embedder] → Query Vector
                              ↓
                    [Vector DB: find top-K similar chunks]
                              ↓
           [LLM: Question + Retrieved Chunks → Answer]
```

Two phases, one goal: give the LLM exactly the right context to answer accurately.

## ⚙️ Setup

We'll use:
- **`sentence-transformers`** — generates embeddings locally (no API key needed for this part!)
- **`chromadb`** — lightweight vector database, runs entirely in-memory
- **`anthropic`** — Claude does the final answer generation

Run the cell below once. It takes ~2 minutes (downloading the embedding model).

In [ ]:
# Install dependencies
!pip install anthropic chromadb sentence-transformers -q

# Load Anthropic API key from Colab Secrets
# Go to: left sidebar → 🔑 Secrets → Add ANTHROPIC_API_KEY
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    # Running locally — set your key here
    os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-YOUR_KEY_HERE'
    print('⚠️  Running locally — update the key above')

import anthropic
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

print('✅ All imports OK')

---

## 🧠 Part 1: What Are Embeddings?

An **embedding** is a list of numbers (a vector) that captures the *meaning* of a piece of text.

Think of it like GPS coordinates for meaning:
- "The dog barked" → `[0.12, -0.45, 0.78, ...]` (384 numbers)
- "The puppy made noise" → `[0.11, -0.43, 0.77, ...]` (very similar!)
- "The stock market crashed" → `[0.89, 0.23, -0.56, ...]` (very different)

Texts with **similar meaning** have vectors that are **close together** in 384-dimensional space.

We measure "closeness" with **cosine similarity** (ranges from -1 to 1, where 1 = identical meaning).

### Why Not Just Use Keyword Search?

Keyword search finds exact words. Embeddings find **meaning**:
- Query: "car"
- Keyword search misses: "automobile", "vehicle", "Tesla"
- Embedding search finds them all because they're semantically close

Let's see this in action:

In [ ]:
# Load a small but powerful embedding model
# all-MiniLM-L6-v2 is fast, free, runs locally — great for learning
print('Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'✅ Model loaded. It produces {embedder.get_sentence_embedding_dimension()}-dimensional vectors.')

# Let's embed some sentences and measure similarity
sentences = [
    'The dog barked at the mailman',         # reference
    'The puppy made loud noises',            # similar meaning
    'A canine was vocalizing aggressively',  # same meaning, different words
    'The stock market dropped 5% today',     # completely different
    'Python is a programming language',      # different domain
]

# Generate embeddings for all sentences
vectors = embedder.encode(sentences)
print(f'\nEach sentence → vector of shape: {vectors[0].shape}')
print(f'Example (first 5 numbers): {vectors[0][:5].round(3)}')

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

reference = vectors[0]  # 'The dog barked at the mailman'

print('Similarity to: "The dog barked at the mailman"')
print('=' * 55)
for i, sentence in enumerate(sentences):
    sim = cosine_similarity([reference], [vectors[i]])[0][0]
    bar = '█' * int(sim * 20)
    print(f'{sim:.3f} {bar:<20} "{sentence[:45]}"')

print()
print('💡 Notice: semantically similar sentences score high even with different words!')

# 💡 EXPERIMENT: Change the reference sentence to something about finance or coding
# and re-run — watch how the similarity scores flip

---

## 🗄️ Part 2: Vector Databases

A vector database stores (text, vector) pairs and can answer one question efficiently:

> *"Given this query vector, which stored vectors are most similar?"*

This is called **Approximate Nearest Neighbor (ANN) search** — it finds the K closest vectors without comparing to every single one (which would be too slow at scale).

### Popular Vector DBs
| Name | Good For |
|------|----------|
| **ChromaDB** | Local dev, prototypes (what we'll use) |
| **Pinecone** | Production, fully managed cloud |
| **Weaviate** | Open-source, production |
| **pgvector** | If you already use PostgreSQL |
| **Qdrant** | High-performance, Rust-based |

For learning, ChromaDB is perfect — no account needed, runs in RAM.

### Our Knowledge Base

We'll create a mini knowledge base about AI concepts — imagine this is your company's internal wiki:

In [ ]:
# Our mock knowledge base — could be scraped from docs, PDFs, wikis, etc.
# In a real system these would be chunked pages from your actual documents.

DOCUMENTS = [
    {
        'id': 'doc_1',
        'text': """LLM Temperature Controls Randomness
The temperature parameter in LLMs controls randomness in token selection.
A temperature of 0 makes the model deterministic — it always picks the most probable next token.
Temperature of 1 is the default, balanced between creativity and coherence.
Temperature above 1 produces more random, creative, sometimes incoherent outputs.
For factual tasks like data extraction, use temperature 0. For creative writing, try 0.8-1.2.""",
        'source': 'AI Fundamentals Guide'
    },
    {
        'id': 'doc_2',
        'text': """Tokens Are the Basic Unit of LLM Processing
LLMs process text as tokens, not characters or words.
One token is roughly 4 characters or 0.75 words in English.
The word 'unbelievable' might be split into ['un', 'believ', 'able'] — 3 tokens.
Models have a context window limit measured in tokens (e.g., 200K tokens for Claude).
Both input and output count toward token usage. API pricing is per token.""",
        'source': 'AI Fundamentals Guide'
    },
    {
        'id': 'doc_3',
        'text': """ReAct Pattern for Agents
ReAct (Reasoning + Acting) is the most common agent architecture.
The agent loop: (1) Reason about what to do, (2) Select a tool/action, (3) Execute the action,
(4) Observe the result, (5) Repeat until the task is done.
The key insight is interleaving reasoning and action — the agent thinks before acting.
This is how Claude's tool use works internally.""",
        'source': 'Agent Architecture Guide'
    },
    {
        'id': 'doc_4',
        'text': """Multi-Agent Systems Use Specialization
Multi-agent systems split complex tasks across specialized agents.
An orchestrator agent breaks down the task and delegates to subagents.
Each subagent is an expert in one domain: research, coding, writing, math, etc.
Benefits: parallelism, specialization, fault isolation.
Challenge: coordinating handoffs between agents without losing context.""",
        'source': 'Agent Architecture Guide'
    },
    {
        'id': 'doc_5',
        'text': """Prompt Engineering Best Practices
Effective prompts are specific, structured, and include examples.
Use XML tags to separate sections: <instructions>, <context>, <output_format>.
Chain-of-thought prompting asks the model to reason step-by-step before answering.
Few-shot prompting includes 2-5 input/output examples to demonstrate the desired format.
Negative examples (what NOT to do) are as valuable as positive examples.""",
        'source': 'Prompt Engineering Guide'
    },
    {
        'id': 'doc_6',
        'text': """Vector Databases Enable Semantic Search
Vector databases store text alongside its embedding vector.
Querying finds the nearest neighbors — most semantically similar stored texts.
Common databases: ChromaDB (local), Pinecone (cloud), Weaviate, Qdrant, pgvector.
Similarity is measured with cosine similarity or dot product.
ANN (Approximate Nearest Neighbor) algorithms make search fast even with millions of vectors.""",
        'source': 'RAG Infrastructure Guide'
    },
    {
        'id': 'doc_7',
        'text': """Agent Memory Types
AI agents have three types of memory:
In-context memory: information in the current conversation window. Fast but limited.
External memory: files, databases the agent reads/writes. Unlimited but requires retrieval.
Semantic memory: vector embeddings of past experiences. Enables fuzzy recall.
Most production agents combine all three for best results.""",
        'source': 'Agent Memory Guide'
    },
    {
        'id': 'doc_8',
        'text': """Tool Use / Function Calling
LLMs can call external tools by outputting structured JSON describing the tool and arguments.
The host application intercepts this output, executes the real function, and returns the result.
This creates a feedback loop: LLM → Tool → Result → LLM → next step.
Tools can be anything: web search, calculators, databases, APIs, code runners.
Anthropic's Claude supports parallel tool calls — multiple tools in one turn.""",
        'source': 'Tool Use Guide'
    },
]

print(f'Knowledge base: {len(DOCUMENTS)} documents')
for doc in DOCUMENTS:
    print(f"  [{doc['id']}] {doc['text'].split(chr(10))[0]}")

---

## 📦 Part 3: Indexing — Embed & Store

The **indexing phase** happens once (or whenever your docs change). We:
1. Take each document
2. Generate its embedding vector
3. Store (text, vector, metadata) in the vector DB

In production, you'd also **chunk** large documents first — split them into 200-500 token pieces. Smaller chunks = more precise retrieval. We're skipping chunking here since our docs are already short.

In [ ]:
# Initialize ChromaDB — runs entirely in RAM
chroma_client = chromadb.Client()

# Create a collection (like a table in SQL)
# We use cosine similarity for comparing vectors
collection = chroma_client.create_collection(
    name='ai_knowledge_base',
    metadata={'hnsw:space': 'cosine'}  # cosine similarity
)

print('Embedding and indexing documents...')

# Embed all documents
texts = [doc['text'] for doc in DOCUMENTS]
doc_ids = [doc['id'] for doc in DOCUMENTS]
metadatas = [{'source': doc['source']} for doc in DOCUMENTS]

# Generate embeddings (batch for efficiency)
embeddings = embedder.encode(texts, show_progress_bar=True)

# Store in ChromaDB
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),  # ChromaDB wants a list of lists
    ids=doc_ids,
    metadatas=metadatas
)

print(f'\n✅ Indexed {collection.count()} documents into vector DB')
print('Each document now has a semantic fingerprint (embedding) stored alongside it.')

---

## 🔍 Part 4: Retrieval — Find Relevant Chunks

At query time:
1. Embed the user's question using the **same model** (crucial!)
2. Ask the vector DB: "what stored vectors are closest to this?"
3. Return the top-K most similar chunks

This is the **R** in RAG.

In [ ]:
def retrieve(question: str, top_k: int = 3) -> list[dict]:
    """Embed the question and retrieve the top_k most relevant chunks."""
    
    # Step 1: Embed the question (same model as indexing!)
    query_vector = embedder.encode([question])[0]
    
    # Step 2: Query the vector DB
    results = collection.query(
        query_embeddings=[query_vector.tolist()],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances']
    )
    
    # Step 3: Package the results
    retrieved_chunks = []
    for doc, metadata, distance in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        # ChromaDB returns distance (lower = more similar for cosine)
        similarity = 1 - distance  # convert to similarity score
        retrieved_chunks.append({
            'text': doc,
            'source': metadata['source'],
            'similarity': similarity
        })
    
    return retrieved_chunks


# Test retrieval — does it find the right docs?
test_question = 'How does temperature affect LLM output?'
chunks = retrieve(test_question, top_k=3)

print(f'Question: "{test_question}"')
print(f'Retrieved {len(chunks)} chunks:\n')
for i, chunk in enumerate(chunks, 1):
    print(f'--- Chunk {i} (similarity: {chunk["similarity"]:.3f}) ---')
    print(f'Source: {chunk["source"]}')
    print(f'{chunk["text"][:200]}...')
    print()

In [ ]:
# Try more questions — see how retrieval works semantically

questions = [
    'What is the ReAct pattern?',
    'How do agents remember things between conversations?',
    'What are the best practices for writing prompts?',
]

for q in questions:
    chunks = retrieve(q, top_k=1)
    top = chunks[0]
    print(f'Q: {q}')
    print(f'   → Best match (sim={top["similarity"]:.3f}): "{top["text"].splitlines()[0]}"')
    print()

# 💡 EXPERIMENT: Try asking about something NOT in the knowledge base
# e.g., 'What is the weather in Paris today?'
# It will still return the most similar docs — but the similarity score will be very low!
# This is how you'd build a 'relevance filter' (discard results below threshold 0.3)

---

## 🤖 Part 5: Generation — Claude Answers with Context

Now the **AG** in RAG — Augmented Generation.

We take the retrieved chunks and inject them into Claude's prompt. Claude can now answer based on YOUR documents, not just its training data.

The key prompt pattern:
```
You are a helpful assistant. Answer using ONLY the provided context.
If the answer is not in the context, say so.

<context>
[retrieved chunks go here]
</context>

Question: [user's question]
```

That final instruction — *"if the answer is not in the context, say so"* — is critical. It prevents hallucination.

In [ ]:
def generate(question: str, chunks: list[dict]) -> str:
    """Send retrieved chunks + question to Claude and return the answer."""
    
    # Build the context block from retrieved chunks
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f'<chunk id="{i}" source="{chunk["source"]}" relevance="{chunk["similarity"]:.2f}">\n'
            f'{chunk["text"]}\n'
            f'</chunk>'
        )
    context_block = '\n\n'.join(context_parts)
    
    # The RAG prompt
    prompt = f"""You are a knowledgeable AI assistant. Answer the question using ONLY the information 
provided in the context below. Be precise and cite which chunk you're drawing from.
If the context does not contain enough information to answer the question, say: 
"I don't have enough information in my knowledge base to answer that."

<context>
{context_block}
</context>

Question: {question}"""
    
    # Call Claude
    client = anthropic.Anthropic()
    response = client.messages.create(
        model='claude-opus-4-5',
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}]
    )
    
    return response.content[0].text


# Test it!
q = 'How does temperature affect LLM output?'
chunks = retrieve(q, top_k=2)
answer = generate(q, chunks)

print(f'Question: {q}')
print(f'\nAnswer:\n{answer}')

---

## 🔗 Part 6: The Complete RAG Pipeline

Let's wire everything into a clean `rag_query()` function and try it on several questions.

In [ ]:
def rag_query(question: str, top_k: int = 3, similarity_threshold: float = 0.3) -> dict:
    """
    Complete RAG pipeline:
    1. Retrieve relevant chunks
    2. Filter by similarity threshold (avoid low-quality retrievals)
    3. Generate answer with Claude
    Returns a dict with the answer and retrieval metadata.
    """
    print(f'\n{"="*60}')
    print(f'❓ Question: {question}')
    print(f'{"="*60}')
    
    # Step 1: Retrieve
    chunks = retrieve(question, top_k=top_k)
    
    # Step 2: Filter low-relevance chunks
    good_chunks = [c for c in chunks if c['similarity'] >= similarity_threshold]
    
    print(f'\n📥 Retrieved {len(chunks)} chunks, {len(good_chunks)} above threshold ({similarity_threshold})')
    for i, c in enumerate(good_chunks, 1):
        print(f'   [{i}] sim={c["similarity"]:.3f} | {c["text"].splitlines()[0][:60]}...')
    
    # Step 3: Generate
    if not good_chunks:
        answer = "I don't have relevant information in my knowledge base to answer that."
    else:
        print('\n🤖 Generating answer with Claude...')
        answer = generate(question, good_chunks)
    
    print(f'\n💬 Answer:\n{answer}')
    
    return {
        'question': question,
        'answer': answer,
        'retrieved_chunks': good_chunks,
    }


# ─── Test the full pipeline ─────────────────────────────────────────

result1 = rag_query('What is the ReAct pattern in AI agents?')

In [ ]:
result2 = rag_query('What are the different types of memory an AI agent can have?')

In [ ]:
# Test with something NOT in our knowledge base
result3 = rag_query('How do I cook a perfect risotto?', similarity_threshold=0.4)

# 💡 Notice: The similarity scores will be very low for out-of-domain questions.
# Setting a higher threshold (0.4) causes the system to say "I don't know"
# instead of hallucinating. This is exactly what you want in production!

---

## ✂️ Part 7: Chunking Strategy (Critical in Production)

Real documents are long. Storing an entire 50-page PDF as one document is bad:
- The retrieved chunk contains mostly irrelevant text
- You burn tokens sending noise to the LLM
- Retrieval precision drops

**Chunking** splits documents into smaller, focused pieces before embedding.

Common strategies:

| Strategy | How | Best For |
|----------|-----|----------|
| **Fixed size** | Split every N characters | Quick & simple |
| **Sentence splitter** | Split at sentence boundaries | General text |
| **Recursive splitter** | Try paragraphs → sentences → words | Structured docs |
| **Semantic chunker** | Split when topic shifts (embedding-based!) | High quality |

Let's implement a simple recursive chunker:

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    """
    Split text into overlapping chunks.
    Overlap ensures context isn't lost at chunk boundaries.
    
    chunk_size: target characters per chunk
    overlap: characters shared between adjacent chunks
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        
        # Don't cut mid-sentence — find the next period or newline
        if end < len(text):
            # Walk back to find a sentence boundary
            for sep in ['. ', '\n', '! ', '? ']:
                boundary = text.rfind(sep, start, end)
                if boundary != -1:
                    end = boundary + len(sep)
                    break
        
        chunks.append(text[start:end].strip())
        start = end - overlap  # overlap for continuity
    
    return [c for c in chunks if len(c) > 20]  # skip tiny leftover chunks


# Demonstrate chunking on a longer document
long_doc = """
Retrieval-Augmented Generation (RAG) was introduced by researchers at Facebook AI Research in 2020.
The paper showed that combining parametric memory (what the LLM learned during training) with 
non-parametric memory (external knowledge retrieved at inference time) produces better results 
than either alone.

The key insight is that LLMs are great reasoners but poor memorizers of facts. 
A RAG system lets the LLM stay lean and fast while delegating factual recall to a vector database.
This is analogous to how humans work: you don't memorize everything — you know where to look it up.

RAG has several variants. Naive RAG is the simple retrieve-then-generate pipeline we built.
Advanced RAG adds query rewriting, re-ranking, and multi-hop retrieval.
Modular RAG treats each component (retriever, reranker, reader) as a swappable module.
Agentic RAG uses an agent that decides when and how many times to retrieve.
"""

chunks = chunk_text(long_doc, chunk_size=250, overlap=50)
print(f'Long doc split into {len(chunks)} chunks:')
for i, chunk in enumerate(chunks, 1):
    print(f'\n--- Chunk {i} ({len(chunk)} chars) ---')
    print(chunk)

# 💡 EXPERIMENT: Change chunk_size to 100 (smaller chunks = more precise retrieval)
# or overlap to 0 (no overlap = chunks may lose context at boundaries)

---

## 🚀 Part 8: Agentic RAG — The Modern Pattern

Naive RAG retrieves once and generates. But what if one retrieval isn't enough?

**Agentic RAG** gives the LLM a `search_knowledge_base` tool. The agent decides:
- When to search
- What to search for (may rewrite the query!)
- Whether the results are good enough or if another search is needed

This is where your previous lessons (Tool Use, ReAct, Agents) combine with RAG.

In [ ]:
# Define a search tool that wraps our retrieval function
search_tool = {
    'name': 'search_knowledge_base',
    'description': (
        'Search the AI knowledge base for relevant information. '
        'Use this whenever you need to answer a factual question about AI concepts, '
        'agents, prompting, or related topics. '
        'You can call this multiple times with different search queries.'
    ),
    'input_schema': {
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'A specific search query to find relevant information'
            },
            'top_k': {
                'type': 'integer',
                'description': 'Number of results to retrieve (default: 3, max: 5)',
                'default': 3
            }
        },
        'required': ['query']
    }
}

def run_agentic_rag(question: str, max_turns: int = 5) -> str:
    """
    Agentic RAG: Claude decides when and what to search.
    The agent loop runs until Claude produces a final answer (no more tool calls).
    """
    client = anthropic.Anthropic()
    
    system_prompt = (
        'You are a knowledgeable AI assistant with access to a knowledge base. '
        'To answer questions accurately, search the knowledge base using the search tool. '
        'You can search multiple times with different queries if needed. '
        'Only answer based on what you retrieve — do not use prior knowledge.'
    )
    
    messages = [{'role': 'user', 'content': question}]
    turns = 0
    
    print(f'\n🤖 Agentic RAG | Question: {question}')
    print('─' * 60)
    
    while turns < max_turns:
        turns += 1
        
        response = client.messages.create(
            model='claude-opus-4-5',
            max_tokens=1024,
            system=system_prompt,
            tools=[search_tool],
            messages=messages
        )
        
        # Add assistant response to conversation
        messages.append({'role': 'assistant', 'content': response.content})
        
        if response.stop_reason == 'end_turn':
            # Agent is done — extract text answer
            for block in response.content:
                if hasattr(block, 'text'):
                    print(f'\n💬 Final Answer:\n{block.text}')
                    return block.text
        
        elif response.stop_reason == 'tool_use':
            # Agent wants to search — execute the tool
            tool_results = []
            for block in response.content:
                if block.type == 'tool_use':
                    query = block.input['query']
                    top_k = block.input.get('top_k', 3)
                    print(f'  🔍 Searching: "{query}" (top_{top_k})')
                    
                    # Execute real retrieval
                    chunks = retrieve(query, top_k=top_k)
                    result_text = '\n\n'.join([
                        f'[Relevance: {c["similarity"]:.2f}]\n{c["text"]}'
                        for c in chunks
                    ])
                    
                    tool_results.append({
                        'type': 'tool_result',
                        'tool_use_id': block.id,
                        'content': result_text
                    })
            
            # Feed results back to the agent
            messages.append({'role': 'user', 'content': tool_results})
    
    return 'Max turns reached'


# Test Agentic RAG
answer = run_agentic_rag(
    'Compare how agents store memory vs how RAG stores knowledge. What is the fundamental difference?'
)

# 💡 EXPERIMENT: Ask a multi-hop question that requires multiple searches:
# 'What are the tools that a ReAct agent can use, and how is that related to function calling?'

---

## 📊 Part 9: Key Concepts Summary

Let's make sure you internalized the core ideas:

In [ ]:
# 🧪 Quick Self-Check — run this cell and answer mentally

questions = [
    ('What is an embedding?',
     'A vector (list of numbers) that encodes the *meaning* of text. Similar meaning = nearby vectors.'),
    
    ('Why must you use the same embedding model for indexing and querying?',
     'Different models produce vectors in different spaces. Comparing them would be meaningless (like comparing GPS coords from different coordinate systems).'),
    
    ('What does the similarity_threshold do in our pipeline?',
     'Filters out chunks that are not relevant enough. Prevents the LLM from getting misleading context.'),
    
    ('What is chunking and why does it matter?',
     'Splitting large docs into smaller pieces. Smaller = more precise retrieval. Too small = loses context. Sweet spot: 200-500 tokens.'),
    
    ('How does Agentic RAG differ from Naive RAG?',
     'In Agentic RAG, the LLM decides when and what to retrieve. It can search multiple times, rewrite queries, and iterate — much more flexible.'),
]

print('🧪 SELF-CHECK')
print('Try to answer each question before revealing the answer.\n')
for i, (q, a) in enumerate(questions, 1):
    print(f'Q{i}: {q}')
    print(f'A:  {a}')
    print()

---

## 🎯 What You Built Today

You now have a working RAG system that:

1. **Generates embeddings** with `sentence-transformers` (semantic vectors)
2. **Stores & retrieves** from ChromaDB (vector database)
3. **Filters** low-relevance chunks with a similarity threshold
4. **Generates answers** using Claude with retrieved context
5. **Handles out-of-scope** questions gracefully (says "I don't know")
6. **Runs as an agent** where Claude decides when to search (Agentic RAG)

---

## 🚀 Experiments to Try

1. **Add your own documents** — put anything into the `DOCUMENTS` list (copy-paste from any article)
2. **Lower the similarity threshold to 0.1** — ask an off-topic question and see if the LLM hallucinates
3. **Compare models** — swap `all-MiniLM-L6-v2` for `all-mpnet-base-v2` (more accurate, slower)
4. **Implement re-ranking** — after retrieval, use Claude to score each chunk's relevance before generating
5. **Multi-hop question** — ask something that requires info from 2+ documents

---

## 📍 Where This Fits in the Curriculum

```
Lesson 1: LLM Fundamentals     ✅
Lesson 2: Prompt Engineering   ✅
Lesson 3: Tool Use             ✅
Lesson 4: First Agent          ✅
Lesson 5: Agent Memory         ✅
Lesson 6: Multi-Agent Systems  ✅
Lesson 7: RAG ← YOU ARE HERE  ✅
Lesson 8: Productionizing AI   ⏳  (evals, observability, cost optimization)
Lesson 9: Capstone Project     ⏳  (build something open-source!)
```

**Next lesson:** We take everything you've built and learn how to ship it — evals, observability, rate limits, and cost optimization. The gap between a prototype and production.

---

*Happy learning, Gourav! — Your AI Tutor*